<a href="https://colab.research.google.com/github/Pranav-Bhatlapenumarthi/BitCamp2023_Art/blob/main/XPINN_trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [30]:
# =====================================================
# Problem Setup
# =====================================================

EPS = 1e-2

def exact_solution(x, t, eps):
    eps_tensor = torch.tensor(eps, device=x.device, dtype=x.dtype)
    denom = 1 - torch.exp(-0.5 / eps_tensor)

    left = torch.exp(-t) * (
        (1 - torch.exp(-(0.5 - x)/eps_tensor)) / denom
        - torch.cos(np.pi*x)
    )

    right = torch.exp(-t) * (
        (-1 - torch.exp(-(x - 0.5)/eps_tensor)) / denom
        - torch.cos(np.pi*x)
    )

    return torch.where(x < 0.5, left, right)

def a_func(x):
    left = -(1 + x*(0.5 - x))
    right = (1 + x*(x - 0.5))
    return torch.where(x <= 0.5, left, right)

In [31]:
# =====================================================
# Subnetwork
# =====================================================

class SubNet(nn.Module):
    def __init__(self, input_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,128),
            nn.Tanh(),
            nn.Linear(128,128),
            nn.Tanh(),
            nn.Linear(128,128),
            nn.Tanh(),
            nn.Linear(128,1)
        )

    def forward(self, x):
        return self.net(x)

In [32]:


# =====================================================
# Layer-Adapted XPINN
# =====================================================

class LayerXPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.left_net = SubNet()
        self.layer_net = SubNet()
        self.right_net = SubNet()

    def tau(self, eps):
        return 2 * eps * np.log(1/eps)

    def forward(self, x, t, eps):

      # ---- Ensure correct shape (N,1) ----
      if x.dim() == 1:
          x = x.unsqueeze(1)
      if t.dim() == 1:
          t = t.unsqueeze(1)

      tau = self.tau(eps)

      # Left region output
      inp_left = torch.cat([x, t], dim=1)
      z_left = self.left_net(inp_left)

      # Right region output
      z_right = self.right_net(inp_left)

      # Layer region output
      xi = (x - 0.5) / eps
      inp_layer = torch.cat([xi, t], dim=1)
      z_layer = self.layer_net(inp_layer)

      # Region masks
      left_mask  = (x < 0.5 - tau)
      layer_mask = (torch.abs(x - 0.5) <= tau)
      right_mask = (x > 0.5 + tau)

      # Assemble solution
      z = torch.where(left_mask, z_left,
          torch.where(layer_mask, z_layer, z_right))

      return z
'''
    def forward(self, x, t, eps):

        tau = self.tau(eps)
        z = torch.zeros_like(x)

        left_mask  = (x < 0.5 - tau)
        layer_mask = (torch.abs(x-0.5) <= tau)
        right_mask = (x > 0.5 + tau)

        if left_mask.any():
          xm = x[left_mask].unsqueeze(1)
          tm = t[left_mask].unsqueeze(1)
          inp = torch.cat([xm, tm], dim=1)
          z[left_mask] = self.left_net(inp)

        if layer_mask.any():
          xl = x[layer_mask].unsqueeze(1)
          tm = t[layer_mask].unsqueeze(1)
          xi = (xl-0.5)/eps
          inp = torch.cat([xi, tm], dim=1)
          z[layer_mask] = self.layer_net(inp)

        if right_mask.any():
            xm = x[right_mask].unsqueeze(1)
            tm = t[right_mask].unsqueeze(1)
            inp = torch.cat([x[right_mask], t[right_mask]], dim=1)
            z[right_mask] = self.right_net(inp)

        return z
'''

'\n    def forward(self, x, t, eps):\n\n        tau = self.tau(eps)\n        z = torch.zeros_like(x)\n\n        left_mask  = (x < 0.5 - tau)\n        layer_mask = (torch.abs(x-0.5) <= tau)\n        right_mask = (x > 0.5 + tau)\n\n        if left_mask.any():\n          xm = x[left_mask].unsqueeze(1)\n          tm = t[left_mask].unsqueeze(1)\n          inp = torch.cat([xm, tm], dim=1)\n          z[left_mask] = self.left_net(inp)\n\n        if layer_mask.any():\n          xl = x[layer_mask].unsqueeze(1)\n          tm = t[layer_mask].unsqueeze(1)\n          xi = (xl-0.5)/eps\n          inp = torch.cat([xi, tm], dim=1)\n          z[layer_mask] = self.layer_net(inp)\n\n        if right_mask.any():\n            xm = x[right_mask].unsqueeze(1)\n            tm = t[right_mask].unsqueeze(1)\n            inp = torch.cat([x[right_mask], t[right_mask]], dim=1)\n            z[right_mask] = self.right_net(inp)\n\n        return z\n'

In [33]:
# =====================================================
# PDE Residual
# =====================================================

def xpinn_residual(model, x, t, eps):

    tau = model.tau(eps)

    x.requires_grad_(True)
    t.requires_grad_(True)

    residual = torch.zeros_like(x)

    left_mask  = (x < 0.5 - tau)
    layer_mask = (torch.abs(x-0.5) <= tau)
    right_mask = (x > 0.5 + tau)

    # -------- Left & Right --------
    for mask in [left_mask, right_mask]:
        if mask.any():
            xm = x[mask]
            tm = t[mask]

            z = model(xm, tm, eps)

            z_x = torch.autograd.grad(z, xm, torch.ones_like(z), create_graph=True)[0]
            z_xx = torch.autograd.grad(z_x, xm, torch.ones_like(z_x), create_graph=True)[0]
            z_t = torch.autograd.grad(z, tm, torch.ones_like(z), create_graph=True)[0]

            delay = torch.where(
                tm < 1,
                exact_solution(xm, tm-1, eps),
                model(xm, tm-1, eps)
            )

            a = a_func(xm)
            reaction = xm*(1-xm)*z

            residual[mask] = -z_t + eps*z_xx + a*z_x - reaction + delay

    # -------- Layer Region --------
    if layer_mask.any():
        xm = x[mask].unsqueeze(1)
        tm = t[mask].unsqueeze(1)

        xi = (xm-0.5)/eps
        xi.requires_grad_(True)

        z = model(xm, tm, eps)

        z_xi = torch.autograd.grad(z, xi, torch.ones_like(z), create_graph=True)[0]
        z_xixi = torch.autograd.grad(z_xi, xi, torch.ones_like(z_xi), create_graph=True)[0]
        z_t = torch.autograd.grad(z, tm, torch.ones_like(z), create_graph=True)[0]

        z_x = z_xi / eps
        z_xx = z_xixi / (eps**2)

        delay = torch.where(
            tm < 1,
            exact_solution(xm, tm-1, eps),
            model(xm, tm-1, eps)
        )

        a = a_func(xm)
        reaction = xm*(1-xm)*z

        residual[layer_mask] = -z_t + eps*z_xx + a*z_x - reaction + delay

    return residual

In [34]:
# =====================================================
# Interface Loss
# =====================================================

def interface_loss(model, eps):

    tau = model.tau(eps)

    t = torch.rand(500,1).to(device)

    xL = torch.ones_like(t)*(0.5 - tau)
    xR = torch.ones_like(t)*(0.5 + tau)

    zL_left = model.left_net(torch.cat([xL,t],dim=1))
    xiL = (xL-0.5)/eps
    zL_layer = model.layer_net(torch.cat([xiL,t],dim=1))

    zR_right = model.right_net(torch.cat([xR,t],dim=1))
    xiR = (xR-0.5)/eps
    zR_layer = model.layer_net(torch.cat([xiR,t],dim=1))

    return torch.mean((zL_left-zL_layer)**2) + \
           torch.mean((zR_right-zR_layer)**2)

In [35]:
# =====================================================
# Training
# =====================================================

def train():

    model = LayerXPINN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(15000):

        x = torch.rand(6000,1).to(device)
        t = 2*torch.rand(6000,1).to(device)

        res = xpinn_residual(model,x,t,EPS)
        loss_pde = torch.mean(res**2)
        loss_int = interface_loss(model,EPS)

        # boundary
        t_bc = torch.rand(1000,1).to(device)
        x0 = torch.zeros_like(t_bc)
        x1 = torch.ones_like(t_bc)

        bc_loss = torch.mean(model(x0,t_bc,EPS)**2) + \
                  torch.mean(model(x1,t_bc,EPS)**2)

        loss = loss_pde + 10*loss_int + 10*bc_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 2000 == 0:
            print(f"Epoch {epoch}, Loss={loss.item():.4e}")

    return model

# =====================================================
# Error Computation
# =====================================================

def compute_error(model):

    x = torch.linspace(0,1,400).unsqueeze(1).to(device)
    t = torch.linspace(0,2,400).unsqueeze(1).to(device)

    X,T = torch.meshgrid(x.squeeze(),t.squeeze(), indexing='ij')
    X = X.reshape(-1,1)
    T = T.reshape(-1,1)

    with torch.no_grad():
        pred = model(X,T,EPS)
        exact = exact_solution(X,T,EPS)

    return torch.max(torch.abs(pred-exact)).item()

In [36]:
# =====================================================
# Run
# =====================================================

if __name__ == "__main__":
    model = train()
    error = compute_error(model)
    print("XPINN Error for ε=1e-2:", error)

RuntimeError: shape mismatch: value tensor of shape [2443, 2443] cannot be broadcast to indexing result of shape [2443]